# Plain TVM API Workflow

This notebook shows the same basic workflow as the project helpers, but without using `tvm_prep` abstractions. Every important step uses TVM's public Python or C++ runtime APIs directly.

The workflow is:

1. Define a small PyTorch vision model.
2. Convert it to TorchScript.
3. Import it into Relay with `relay.frontend.from_pytorch`.
4. Compile it for a selected target with `relay.build`.
5. Export `model.so`, `model.json`, and `model.params` manually.
6. Load and run the artifacts with TVM's Python graph executor.
7. Build a minimal C++ program that uses TVM runtime APIs directly.
8. Run the same compiled artifacts from C++.

The executable path in this notebook targets local `x86_64` Linux so it can run inside the devcontainer. For Raspberry Pi, use the target notes near the end and run the compiled artifacts on the Pi.

In [1]:
from __future__ import annotations

import os
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import torch
import tvm
from tvm import relay
from tvm.contrib import graph_executor

WORK_DIR = Path(tempfile.mkdtemp(prefix='plain-tvm-api-'))
ARTIFACT_DIR = WORK_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('TVM:', tvm.__version__)
print('Work directory:', WORK_DIR)

TVM: 0.19.0
Work directory: /tmp/plain-tvm-api-o5d2jbmn


## Define A Small Vision Model

This model averages the RGB channels and scores red, green, and blue. It is tiny, deterministic, and still uses an image-shaped tensor: `(N, C, H, W) = (1, 3, 32, 32)`.

A red input should produce class `0`. That gives us a simple correctness check for both Python and C++ runtimes.

In [2]:
class RGBMeanClassifier(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.classifier = torch.nn.Linear(3, 3, bias=False)
        with torch.no_grad():
            self.classifier.weight.copy_(
                torch.tensor(
                    [
                        [4.0, -1.0, -1.0],
                        [-1.0, 4.0, -1.0],
                        [-1.0, -1.0, 4.0],
                    ],
                    dtype=torch.float32,
                )
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        channel_means = x.mean(dim=(2, 3))
        return self.classifier(channel_means)


torch.manual_seed(0)
torch.set_grad_enabled(False)

input_name = 'input0'
input_shape = (1, 3, 32, 32)
model = RGBMeanClassifier().eval()

red_input = np.zeros(input_shape, dtype='float32')
red_input[:, 0, :, :] = 1.0

with torch.no_grad():
    torch_output = model(torch.from_numpy(red_input)).numpy()

print('PyTorch logits:', torch_output)
assert int(torch_output.argmax()) == 0

PyTorch logits: [[ 4. -1. -1.]]


## Import The Model Into Relay

TVM's PyTorch frontend expects a TorchScript module plus a shape list. The shape list tells TVM the graph input name and static input shape to compile for.

In [3]:
example_input = torch.zeros(input_shape, dtype=torch.float32)
scripted = torch.jit.trace(model, example_input).eval()
shape_list = [(input_name, input_shape)]

mod, params = relay.frontend.from_pytorch(scripted, shape_list)

print(type(mod).__name__)
print('Parameter tensors:', len(params))
assert 'main' in mod.get_global_vars()[0].name_hint or mod['main'] is not None

IRModule
Parameter tensors: 1


## Compile For A Target

`target` controls the architecture TVM generates code for. This notebook uses local `x86_64` so we can execute the result immediately in the devcontainer.

For Raspberry Pi 4 64-bit, the target string would be:

```python
target = tvm.target.Target('llvm -mtriple=aarch64-linux-gnu -mcpu=cortex-a72 -mattr=+neon')
target_host = tvm.target.Target('llvm -mtriple=aarch64-linux-gnu')
```

You can compile that in the devcontainer, but you must run the resulting `model.so` on an AArch64 device such as a Raspberry Pi, not on the x86_64 host.

In [4]:
target = tvm.target.Target('llvm -mtriple=x86_64-linux-gnu -mcpu=x86-64')

with tvm.transform.PassContext(opt_level=3):
    compiled = relay.build(mod, target=target, params=params)

print(type(compiled).__name__)

One or more operators have not been tuned. Please tune your model for better performance. Use DEBUG logging level to see more details.


GraphExecutorFactoryModule


## Export Graph Executor Artifacts Manually

The graph executor needs three files:

- `model.so`: compiled kernels for the target architecture
- `model.json`: graph executor graph
- `model.params`: serialized model parameters

The project helpers also write `metadata.json`, but this notebook intentionally shows the raw TVM artifact flow.

In [5]:
lib_path = ARTIFACT_DIR / 'model.so'
graph_path = ARTIFACT_DIR / 'model.json'
params_path = ARTIFACT_DIR / 'model.params'

compiled.export_library(str(lib_path))
graph_path.write_text(compiled.get_graph_json(), encoding='utf-8')
params_path.write_bytes(relay.save_param_dict(compiled.get_params()))

for path in sorted(ARTIFACT_DIR.iterdir()):
    print(path.name, path.stat().st_size, 'bytes')

assert lib_path.exists()
assert graph_path.exists()
assert params_path.exists()

model.json 1580 bytes
model.params 142 bytes
model.so 40488 bytes


## Run With TVM Python Runtime APIs

This cell loads the artifact files with TVM directly, creates a graph executor, sets the input tensor, runs inference, and reads output tensor `0`.

In [6]:
loaded_lib = tvm.runtime.load_module(str(lib_path))
graph_json = graph_path.read_text(encoding='utf-8')
param_bytes = params_path.read_bytes()

dev = tvm.cpu(0)
module = graph_executor.create(graph_json, loaded_lib, dev)
module.load_params(param_bytes)
module.set_input(input_name, red_input)
module.run()
tvm_output = module.get_output(0).numpy()

print('TVM Python logits:', tvm_output)
assert int(tvm_output.argmax()) == 0
np.testing.assert_allclose(tvm_output, torch_output, rtol=1e-5, atol=1e-5)
print('PASS: Python runtime output matches PyTorch')

TVM Python logits: [[ 4. -1. -1.]]
PASS: Python runtime output matches PyTorch


## Build And Run A Minimal C++ TVM Runtime Program

The C++ program below uses TVM runtime APIs directly. It loads the same `model.so`, `model.json`, and `model.params`, creates an input tensor filled with a red image, runs the graph executor, and prints the top class.

This is intentionally generated into the temporary work directory so the notebook remains self-contained.

In [7]:
cpp_source = r'''
#include <dlpack/dlpack.h>
#include <tvm/runtime/module.h>
#include <tvm/runtime/ndarray.h>
#include <tvm/runtime/packed_func.h>
#include <tvm/runtime/registry.h>

#include <algorithm>
#include <fstream>
#include <iostream>
#include <iterator>
#include <stdexcept>
#include <string>
#include <vector>

std::string ReadAll(const std::string& path) {
  std::ifstream file(path, std::ios::binary);
  if (!file) {
    throw std::runtime_error("Cannot open " + path);
  }
  return std::string((std::istreambuf_iterator<char>(file)), std::istreambuf_iterator<char>());
}

int main(int argc, char** argv) {
  try {
    if (argc != 2) {
      std::cerr << "Usage: plain_tvm_runner ARTIFACT_DIR\n";
      return 2;
    }
    std::string artifact_dir = argv[1];
    tvm::runtime::Module lib = tvm::runtime::Module::LoadFromFile(artifact_dir + "/model.so");
    std::string graph_json = ReadAll(artifact_dir + "/model.json");
    std::string params = ReadAll(artifact_dir + "/model.params");

    auto fcreate = tvm::runtime::Registry::Get("tvm.graph_executor.create");
    if (!fcreate) {
      throw std::runtime_error("tvm.graph_executor.create not found");
    }
    tvm::runtime::Module graph = (*fcreate)(graph_json, lib, static_cast<int>(kDLCPU), 0);
    graph.GetFunction("load_params")(tvm::runtime::String(params));

    constexpr int N = 1;
    constexpr int C = 3;
    constexpr int H = 32;
    constexpr int W = 32;
    std::vector<float> input(N * C * H * W, 0.0f);
    for (int i = 0; i < H * W; ++i) {
      input[i] = 1.0f;
    }

    tvm::runtime::NDArray input_nd = tvm::runtime::NDArray::Empty(
        {N, C, H, W}, DLDataType{kDLFloat, 32, 1}, DLDevice{kDLCPU, 0});
    input_nd.CopyFromBytes(input.data(), input.size() * sizeof(float));

    graph.GetFunction("set_input")("input0", input_nd);
    graph.GetFunction("run")();
    tvm::runtime::NDArray output_nd = graph.GetFunction("get_output")(0);

    std::vector<float> output(3);
    output_nd.CopyToBytes(output.data(), output.size() * sizeof(float));
    int top_id = static_cast<int>(std::distance(output.begin(), std::max_element(output.begin(), output.end())));

    std::cout << "logits=";
    for (float value : output) {
      std::cout << value << " ";
    }
    std::cout << "\ntop_id=" << top_id << "\n";
    return top_id == 0 ? 0 : 1;
  } catch (const std::exception& exc) {
    std::cerr << "Error: " << exc.what() << "\n";
    return 1;
  }
}
'''

cpp_path = WORK_DIR / 'plain_tvm_runner.cpp'
runner_path = WORK_DIR / 'plain_tvm_runner'
cpp_path.write_text(cpp_source, encoding='utf-8')
print(cpp_path)

/tmp/plain-tvm-api-o5d2jbmn/plain_tvm_runner.cpp


In [8]:
compile_cmd = [
    'g++',
    '-std=c++17',
    str(cpp_path),
    '-I/opt/tvm/include',
    '-I/opt/tvm/3rdparty/dlpack/include',
    '-I/opt/tvm/3rdparty/dmlc-core/include',
    '-L/opt/tvm/build',
    '-ltvm_runtime',
    '-Wl,-rpath,/opt/tvm/build',
    '-o',
    str(runner_path),
]
subprocess.run(compile_cmd, check=True)

env = os.environ.copy()
env['LD_LIBRARY_PATH'] = f"/opt/tvm/build:{env.get('LD_LIBRARY_PATH', '')}"
result = subprocess.run(
    [str(runner_path), str(ARTIFACT_DIR)],
    check=True,
    text=True,
    capture_output=True,
    env=env,
)

print(result.stdout)
assert 'top_id=0' in result.stdout
print('PASS: C++ runtime predicted red class')

In file included from /opt/tvm/include/tvm/runtime/container/base.h:28,
                 from /opt/tvm/include/tvm/runtime/container/string.h:29,
                 from /opt/tvm/include/tvm/runtime/module.h:31,
                 from /tmp/plain-tvm-api-o5d2jbmn/plain_tvm_runner.cpp:3:
/opt/tvm/include/tvm/runtime/logging.h:594: warning: "LOG" redefined
  594 | #define LOG(level) LOG_##level
      | 
In file included from /opt/tvm/3rdparty/dmlc-core/include/dmlc/io.h:15,
                 from /opt/tvm/include/tvm/runtime/module.h:29:
/opt/tvm/3rdparty/dmlc-core/include/dmlc/./logging.h:263: note: this is the location of the previous definition
  263 | #define LOG(severity) LOG_##severity.stream()
      | 
/opt/tvm/include/tvm/runtime/logging.h:597: warning: "LOG_FATAL" redefined
  597 | #define LOG_FATAL ::tvm::runtime::detail::LogFatal(__FILE__, __LINE__).stream()
      | 
/opt/tvm/3rdparty/dmlc-core/include/dmlc/./logging.h:257: note: this is the location of the previous definition
  25

logits=4 -1 -1 
top_id=0

PASS: C++ runtime predicted red class


## What Changes For Raspberry Pi?

The TVM Python calls are the same, but the target is different and the output must run on the Pi:

```python
target = tvm.target.Target('llvm -mtriple=aarch64-linux-gnu -mcpu=cortex-a72 -mattr=+neon')
target_host = tvm.target.Target('llvm -mtriple=aarch64-linux-gnu')

with tvm.transform.PassContext(opt_level=3):
    compiled = relay.build(mod, target=target, target_host=target_host, params=params)
```

Export the same three graph executor files. Copy them to the Raspberry Pi. Then run with Python or C++ on the Pi using TVM runtime libraries built for AArch64. The x86_64 devcontainer cannot execute the AArch64 `model.so`.